# LangChain: Models, Prompts and Output Parsers
## Outline
- Direct API calls to OpenAI
- API calls through LangChain:
    - Prompts
    - Models
    - Output parsers

In [ ]:
#!pip install python-dotenv
#!pip install openai

In [10]:
# load api key and relevant libraries

import openai
import os


from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())


token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.github.ai/inference"
llm_model = "openai/gpt-4.1"

client = openai.OpenAI(
    base_url=endpoint,
    api_key=token,
)

## Chat API : OpenAI
Let's start with a direct API calls to OpenAI.

In [6]:
# helper function to get a completion from OpenAI API
def get_completion(prompt, model=llm_model):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content

In [3]:
get_completion("What is 2+2 ? ")

'2 + 2 = **4**'

In [6]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""


style = """American English \
in a calm and respectful tone
"""

In [8]:
prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style} .
text: ```{customer_email}```
"""

# print(prompt)

In [9]:
reponse = get_completion(prompt)
print(reponse)

I'm really frustrated that my blender lid came off and splattered smoothie all over my kitchen walls. To make things worse, the warranty doesn't cover the cost of cleaning up my kitchen. I could really use your help right now.


## Chat API : LangChain
Let's try how we can do the same using LangChain.

In [ ]:
# !pip install --upgrade langchain  # install or upgrade langchain
# !pip install langchain-openai

### Model

In [2]:
from langchain_openai import ChatOpenAI

In [11]:
from langchain.messages import HumanMessage
# To control the randomness and creativity of the generated
# text by an LLM, use temperature = 0.0
chat = ChatOpenAI(temperature=0.0, model = llm_model, api_key=token,                   # GitHub token
    base_url=endpoint
)
response = chat.invoke([
    HumanMessage(content="Write a haiku about recursion in programming.")
])

response.content

'Function calls itself,  \nDepths unfold, patterns repeat—  \nEnd case brings return.'

### Prompt template

In [12]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [13]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_template(template_string)


In [14]:
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

In [15]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [16]:
customer_style = """American English \
in a calm and respectful tone
"""

In [17]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [18]:
customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

In [20]:
print(type(customer_messages))
print(type(customer_messages[0]))
print(customer_messages[0])

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>
content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}


In [26]:
# Call the LLM to translate to the style of the customer message
customer_response = chat.invoke(customer_messages)  # AIMessage object
customer_response.content

"I'm really frustrated that my blender lid came off and splattered smoothie all over my kitchen walls. To make things worse, the warranty doesn't cover the cost of cleaning up my kitchen. I could really use your help right now."

In [ ]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

service_style_pirate = """\
a polite tone \
that speaks in English Pirate and just return conversion not additional text \
"""  # here i am asking to just return the conversion as model is adding additional text

In [37]:
service_messages = prompt_template.format_messages(style=service_style_pirate,text=service_reply)
# print(service_messages[0].content)

In [38]:
service_response = chat.invoke(service_messages)
service_response.content

'Ahoy, matey! Alas, the warranty shan’t be coverin’ the cost o’ cleanin’ yer galley, for ’tis yer own misadventure that led to the mess—ye forgot to secure the lid afore ye set sail with the blender. Fair winds to ye, and may fortune favor ye next time!'

## Output Parser
Let's start with defining how we would like the LLM output to look like:

In [ ]:
{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

In [11]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [42]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]


In [ ]:
messages = prompt_template.format_messages(text=customer_review)
chat = ChatOpenAI(temperature=0.0, model=llm_model , api_key=token,
    base_url=endpoint
)
response = chat.invoke(messages)
print(response.content)


{
  "gift": true,
  "delivery_days": 2,
  "price_value": ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]
}


In [ ]:
type(response.content)

## response.content is not a dictionary yet , it is a string in JSON format

str

### Parse the LLM output string into a Python dictionary

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI  # Replace with your LLM

In [ ]:
# Step 1: Define JSON schema 
json_schema = {
    "type": "object",
    "properties": {
        "gift": {
            "type": "boolean",
            "description": "Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown."
        },
        "delivery_days": {
            "type": "integer",
            "description": "How many days did it take for the product to arrive? If this information is not found, output -1."
        },
        "price_value": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Extract any sentences about the value or price, and output them as a list."
        }
    },
    "required": ["gift", "delivery_days", "price_value"]
}

# Step 2: Create parser 
output_parser = JsonOutputParser(json_schema=json_schema)

# step 3 : get format instructions
format_instructions = output_parser.get_format_instructions()
print("Format Instructions:\n", format_instructions)

Format Instructions:
 Return a JSON object.


In [19]:
# Step 4: Create prompt template 
prompt = ChatPromptTemplate.from_template("""
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price, and output them as a list.

text: {text}

{format_instructions}
""")


chat = ChatOpenAI(temperature=0.0, model=llm_model, api_key=token,
    base_url=endpoint
)



response = chat.invoke(prompt.format_messages(
    text=customer_review,format_instructions=format_instructions))


output_dict = output_parser.parse(response.content)
print(type(output_dict))  # <class 'dict'>
print(output_dict.get("delivery_days")) 

<class 'dict'>
2


In [22]:
# in this does not neet to use parse method of parser , direct reponse of llm is parsed
chain = prompt | chat | output_parser
response = chain.invoke({"text": customer_review,"format_instructions": format_instructions})
type(response)
print(response.get("delivery_days"))

2
